In [33]:
import numpy as np
import pandas as pd
from netCDF4 import Dataset as NetCDFFile
from scipy.interpolate import RegularGridInterpolator
import os

In [41]:
##USEFUL FUNCTIONS

def normalize_array(data):
    vmax = np.amax(data[np.nonzero(data)])
    vmin = np.amin(data[np.nonzero(data)])
    range_data = vmax - vmin
    
    normalized_data = (data - vmin)/range_data
    threshold = - 1./range_data
    data = np.maximum(normalized_data,threshold)
    data[data == threshold] = -1
    
    return data

def round_days(data):
    l = len(data)
    for i in range(l):
        data[i] = int(data[i])
    data = data.astype(int)
    return data

def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

# The boundary format is low_lat, left_lon, high_lat and right_lon IN INDICES !!! Not in degrees
# The box is centered on the sensor and 20x20 degrees
def boundaries(coordinates, lat, lon, size):
    nb_sensors = len(coordinates)
    boundaries = np.ones((nb_sensors, 4)).astype(int)
    for i in range(nb_sensors):
        lat_sensor = float(coordinates[i][2])
        lon_sensor = float(coordinates[i][1])
        boundaries[i, 0] = find_nearest(lat, lat_sensor - size)
        boundaries[i, 1] = find_nearest(lon, lon_sensor - size)
        boundaries[i, 2] = find_nearest(lat, lat_sensor + size)
        boundaries[i, 3] = find_nearest(lon, lon_sensor + size)
    return boundaries

def find_bic_pixel(box, coords):
    start_lat = box[0]
    start_lon = box[1]
    end_lat = box[2]
    end_lon = box[3]

    coord_low_lat = start_lat*180/361-90
    coord_left_lon = start_lon*360/576-180
    coord_up_lat = end_lat*180/361-90
    coord_right_lon = end_lon*360/576-180

    step_lat = 180/(361*16)
    step_lon = 360/(576*16)

    lats = np.arange(coord_low_lat, coord_up_lat, step_lat)
    lons = np.arange(coord_left_lon, coord_right_lon, step_lon)

    return find_nearest(lats, float(coords[2]))/16, find_nearest(lons, float(coords[1]))/16

def array_centered_on_sensor(xco2_array, boundaries):
    return xco2_array[boundaries[0]:boundaries[2], boundaries[1]:boundaries[3]]

def create_date_column(year, day):
    assert len(year) == len(day)
    real_date = []
    real_date_str = []
    for i in range(len(year)):
        try:
            raw_date = datetime.strptime(str(year[i])+str(day[i]), "%Y%j")
        except:
            print("Error with date", year[i], day[i])
        real_date.append(raw_date)
        real_date_str.append(raw_date.strftime("%Y%m%d"))
    return real_date, real_date_str
    
def buil_TCCON_mean(file):
    data = NetCDFFile(file, 'r')
    xco2 = np.squeeze(np.asarray(data.variables['xco2'][:]))
    xco2_error = np.squeeze(np.asarray(data.variables['xco2_error'][:]))
    year = np.squeeze(np.asarray(data.variables['year'][:]))
    day = np.squeeze(np.asarray(data.variables['day'][:]))
    real_date, real_date_str = create_date_column(year, day)
    df = pd.DataFrame({'xco2_ppm':xco2, 'xco2_ppm_error':xco2_error, 'real_date': real_date, 'real_date_str':real_date_str})
    df = df.groupby('real_date_str').mean()
    df.reset_index(inplace=True)
    df.sort_values(by=['real_date'], inplace=True)
    df.reset_index(inplace=True, drop=True)
    df = df.astype({'real_date_str': 'str'})
    return df
    

def filter_dates(file):
    data = NetCDFFile(file, 'r')
    year = np.squeeze(np.asarray(data.variables['year'][:]))
    day = np.squeeze(np.asarray(data.variables['day'][:]))
    fvsi = np.squeeze(np.asarray(data.variables['fvsi'][:]))
    real_date, real_date_str = create_date_column(year, day)
    df = pd.DataFrame({'fvsi':fvsi, 'real_date': real_date, 'real_date_str':real_date_str})
    df = df.groupby('real_date_str').mean()
    df.reset_index(inplace=True)
    df.sort_values(by=['real_date'], inplace=True)
    df.reset_index(inplace=True, drop=True)
    df = df.astype({'real_date_str': 'str'})
    return df

def get_lat_lon(site):
    lon = site['long_deg'][0].data.item()
    lat = site['lat_deg'][0].data.item()
    return lon, lat

def unnormalize(data, original_data, bound_left, bound_right, bound_bottom, bound_top):
    original_data = original_data[bound_left:bound_right, bound_bottom:bound_top]
    v_max = np.amax(original_data[np.nonzero(original_data)])
    v_min = np.amin(original_data[np.nonzero(original_data)])
    h = len(data)
    w = len(data[0])
    z = np.copy(data)
    for i in range(h):
        for j in range(w):
            z[i,j] = data[i, j]*(v_max - v_min) + v_min
    return z

def unnormalize2(data, original_data):
    bound_right = int(288/16)
    bound_left = int(227/16)
    bound_bottom = int(301/16)
    bound_top = int(342/16)
    original_data = original_data[bound_left:bound_right, bound_bottom:bound_top]
    v_max = np.amax(original_data[np.nonzero(original_data)])
    v_min = np.amin(original_data[np.nonzero(original_data)])
    h = len(data)
    w = len(data[0])
    z = np.copy(data)
    for i in range(h):
        for j in range(w):
            z[i,j] = data[i, j]*(v_max - v_min) + v_min
    return z

def process_dates(row):
    stri = row['real_date_str']
    row['real_date_str'] = re.sub(r'^.*?\'','\'', stri)  
    return row

##SR best pixel
def get_best_pixel(array, boundaries, coords):
    h, w = array.shape
    lat_options = np.linspace(boundaries[0], boundaries[2], h)
    lon_options = np.linspace(boundaries[1], boundaries[3], w)
    best_lat = find_nearest(lat_options, float(coords[2]))
    best_lon = find_nearest(lon_options, float(coords[1]))
    #print(sensor_coord[i], lat_options[best_lat], lon_options[best_lon])
    return array[best_lat, best_lon]*1e6

##SR_SR best pixel
def get_best_pixel2(array, boundaries, coords):
    h, w = array.shape
    bound_right = (288/512)*boundaries[3] + (224/512)*boundaries[1]
    bound_left = (227/512)*boundaries[3] + (285/512)*boundaries[1]
    bound_bottom = (301/640)*boundaries[2] + (339/640)*boundaries[0]
    bound_top = (342/640)*boundaries[2] + (298/640)*boundaries[0]
    lat_options = np.linspace(bound_left, bound_right, h)
    lon_options = np.linspace(bound_bottom, bound_top, w)
    best_lat = find_nearest(lat_options, float(coords[2]))
    best_lon = find_nearest(lon_options, float(coords[1]))
    #print(sensor_coord[i], lat_options[best_lat], lon_options[best_lon])
    return array[best_lat, best_lon]*1e6

##LR best pixel
def get_best_grid_point_fusion(file, coord_array):
    lats = float(coord_array[2])
    lons = float(coord_array[1])
    xco2 = np.asarray(file.variables['XCO2'][:])
    lat_options = np.asarray(file.variables['Lat'][:])
    lon_options = np.asarray(file.variables['Lon'][:])
    best_lat = find_nearest(lat_options, lats)
    best_lon = find_nearest(lon_options, lons)
    return xco2[best_lat, best_lon]

#def remove_date_without_observations(dataframe):
    

In [6]:
coord_array = np.load('../Paper/coordinates_2.npy', allow_pickle=True)
coordinates = pd.DataFrame(coord_array, columns=['site', 'lon', 'lat'])

file = NetCDFFile('../../RDS/OCO-2/oco2_GEOS_L3CO2_day_20150101_B10206Ar.nc4', 'r')
lat = np.asarray(file.variables['lat'][:])
lon = np.asarray(file.variables['lon'][:])

boundary_array = boundaries(coord_array, lat, lon, 10)

In [17]:
coord_test

array(['sp20140406_20200925', '11.920000076293945', '78.91999816894531'],
      dtype='<U32')

In [32]:
box_test = boundary_array[2]
coord_test = coord_array[2]

find_bic_pixel(box_test, coord_test)

(21.1875, 16.4375)

In [43]:
file = NetCDFFile('../../RDS/data_fusion/2015/2015-01-05.nc')
site = 'ae20120522_20181031'

coord = coord_array[np.where(coord_array==site)[0][0]]

get_best_grid_point_fusion(file, coord)

398.3404

In [38]:
file = np.load('../../RDS/OCO-2/centered_arrays/ae20120522_20181031_20150105_lr.npy')
site = 'ae20120522_20181031'

h, w = file.shape
x  = np.arange(0, h)
y  = np.arange(0, w)
interpolator = RegularGridInterpolator((x, y), file, method='cubic')

box = boundary_array[np.where(coord_array==site)[0][0]]
coord = coord_array[np.where(coord_array==site)[0][0]]

interpolator([find_bic_pixel(box, coord)])[0], file[int(h/2), int(w/2)]

(0.000397909935269284, 0.00039788399590179324)